In [ ]:
!pip install -U langgraph langchain langchain-groq langchain-tavily pygraphviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.0 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.17
    Uninstalling langchain-1.3.17:
      Successfully uninstalled langchain-1.3.17


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

In [ ]:
from langchain_core.messages import HumanMessage, AnyMessage, SystemMessage, ToolMessage
from langchain_tavily import TavilySearch

In [ ]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API KEY here: ")


Enter your GROQ API KEY here: ··········


In [ ]:
if "TAVILY_API_KEY" not in os.environ:
  os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your TAVILLY API KEY here:")

Enter your TAVILLY API KEY here:··········


In [ ]:
tool = TavilySearch(max_results=2)
print(type(tool))
print(tool.name)

<class 'langchain_tavily.tavily_search.TavilySearch'>
tavily_search


In [ ]:
from uuid import uuid4
from langchain_core.messages import AnyMessage

def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]):
    # Create a copy of the left list to build the merged list
    merged = left.copy()

    # Store IDs of messages currently in 'merged' for efficient lookup
    merged_ids = {msg.id for msg in merged if msg.id}

    for message in right:
        if not message.id:
            message.id = str(uuid4()) # Ensure all messages have an ID

        if message.id in merged_ids:
            # If message with this ID exists, update it
            for i, existing_msg in enumerate(merged):
                if existing_msg.id == message.id:
                    merged[i] = message
                    break
        else:
            # If message is new, append it
            merged.append(message)
            merged_ids.add(message.id)

    return merged

class AgentState(TypedDict):
  messages: Annotated[list[AnyMessage], reduce_messages]

In [ ]:
# MANUAL HUMAN APPROVAL

In [ ]:
class Agent:
  def __init__(self, model, tools, system="", checkpointer=None):
    self.system = system
    graph = StateGraph(AgentState)
    graph.add_node("llm", self.call_grok)
    graph.add_node("action", self.take_action)
    graph.add_conditional_edges("llm", self.exists_action, {True: "action",False: END})
    graph.add_edge("action", "llm")
    graph.set_entry_point("llm")
    self.graph = graph.compile(
        checkpointer=checkpointer,
        interrupt_before=["action"]
    )
    self.tools = {t.name: t for t in tools}
    self.model = model.bind_tools(tools)

  def call_grok(self, state: AgentState):
    messages = state["messages"]
    if self.system:
      messages = [SystemMessage(content=self.system)] + messages
    message = self.model.invoke(messages)
    return {"messages":[message]}

  def exists_action(self, state: AgentState):
    print(state)
    result = state['messages'][-1]
    return len(result.tool_calls) > 0

  def take_action(self, state: AgentState):
    tool_calls = state["messages"][-1].tool_calls
    results = []
    for t in tool_calls:
      print(f"Calling: {t}")
      result = self.tools[t['name']].invoke(t['args'])
      results.append(ToolMessage(tool_call_id=t['id'], name= t['name'], content=str(result)))
    print("Back to Model!")
    return {"messages":results}




In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
checkpointer = InMemorySaver()
memory = InMemoryStore()

In [ ]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

abot = Agent(model, [tool], system=prompt, checkpointer=checkpointer)

In [ ]:
messages = [HumanMessage(content="What is weather in SF")]

In [ ]:
thread= {"configurable" : {"thread_id": "1"}}

In [ ]:
for event in abot.graph.stream({"messages":messages}, thread):
  for v in event.values():
    print(v)

{'messages': [HumanMessage(content='What is weather in SF', additional_kwargs={}, response_metadata={}, id='f7b48d12-d75c-409e-8bf7-5914018495b8'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is weather in SF". Likely they want current weather in San Francisco. Need to search. Use search query "current weather San Francisco".', 'tool_calls': [{'id': 'fc_ac43041d-92f6-4154-ac1f-c0e88b140100', 'function': {'arguments': '{"query":"current weather San Francisco","search_depth":"fast","time_range":"day"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 386, 'total_tokens': 469, 'completion_time': 0.175564652, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.017202649, 'prompt_tokens_details': None, 'queue_time': 0.402883958, 'total_time': 0.192767301}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_dema

In [ ]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='What is weather in SF', additional_kwargs={}, response_metadata={}, id='f7b48d12-d75c-409e-8bf7-5914018495b8'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is weather in SF". Likely they want current weather in San Francisco. Need to search. Use search query "current weather San Francisco".', 'tool_calls': [{'id': 'fc_ac43041d-92f6-4154-ac1f-c0e88b140100', 'function': {'arguments': '{"query":"current weather San Francisco","search_depth":"fast","time_range":"day"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 386, 'total_tokens': 469, 'completion_time': 0.175564652, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.017202649, 'prompt_tokens_details': None, 'queue_time': 0.402883958, 'total_time': 0.192767301}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'se

In [ ]:
abot.graph.get_state(thread).next

('action',)

In [ ]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search', 'args': {'query': 'current weather San Francisco', 'search_depth': 'fast', 'time_range': 'day'}, 'id': 'fc_ac43041d-92f6-4154-ac1f-c0e88b140100', 'type': 'tool_call'}
Back to Model!
{'messages': [ToolMessage(content='{\'query\': \'current weather San Francisco\', \'follow_up_questions\': None, \'answer\': None, \'images\': [], \'results\': [{\'url\': \'https://www.weatherandradar.in/weather/san-francisco/12350786\', \'title\': \'Current weather for San Francisco\', \'content\': "Weather San Francisco\\n\\nWeather News\\n\\nEditor\'s Pick\\n\\nDiscover the app\\n\\nWeather widget\\n\\nContact us\\n\\nCordoba · Argentina  ср, 19.08 05:25\\n\\n# Weather San Francisco\\n\\n now \\n90 Min. Weather\\n\\nmostly cloudy\\n\\nSunrise and sunset 07:39 am  06:44 pm\\n\\n06:00\\n\\n5\\n\\n°\\n\\nProbability of precipitation\\n\\n10 %\\n\\n07:00\\n\\n4\\n\\n°\\n\\nProbability of precipitation\\n\\n10 %\\n\\n08:00\\n\\n4\\n\\n°\\n\\nProbability of precipitation\\n\\

In [ ]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='What is weather in SF', additional_kwargs={}, response_metadata={}, id='f7b48d12-d75c-409e-8bf7-5914018495b8'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is weather in SF". Likely they want current weather in San Francisco. Need to search. Use search query "current weather San Francisco".', 'tool_calls': [{'id': 'fc_ac43041d-92f6-4154-ac1f-c0e88b140100', 'function': {'arguments': '{"query":"current weather San Francisco","search_depth":"fast","time_range":"day"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 386, 'total_tokens': 469, 'completion_time': 0.175564652, 'completion_tokens_details': {'reasoning_tokens': 35}, 'prompt_time': 0.017202649, 'prompt_tokens_details': None, 'queue_time': 0.402883958, 'total_time': 0.192767301}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'se

In [ ]:
abot.graph.get_state(thread).next

()

In [ ]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
while abot.graph.get_state(thread).next:
    print("\n", abot.graph.get_state(thread),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='bd2d0be6-224b-41a3-a4e3-d496193e5064'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "Whats the weather in LA?" Likely they want current weather. Need to search. Use search query "current weather Los Angeles". Use time_range maybe day.', 'tool_calls': [{'id': 'fc_9f79dfc4-cc7c-4290-8080-3cfe27ae23ac', 'function': {'arguments': '{"query":"current weather Los Angeles","search_depth":"fast","time_range":"day"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 387, 'total_tokens': 472, 'completion_time': 0.178245248, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.021059483, 'prompt_tokens_details': None, 'queue_time': 0.409250204, 'total_time': 0.199304731}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier':

In [ ]:
result = abot.graph.invoke({"messages": messages}, thread)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='bd2d0be6-224b-41a3-a4e3-d496193e5064'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "Whats the weather in LA?" Likely they want current weather. Need to search. Use search query "current weather Los Angeles". Use time_range maybe day.', 'tool_calls': [{'id': 'fc_9f79dfc4-cc7c-4290-8080-3cfe27ae23ac', 'function': {'arguments': '{"query":"current weather Los Angeles","search_depth":"fast","time_range":"day"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 387, 'total_tokens': 472, 'completion_time': 0.178245248, 'completion_tokens_details': {'reasoning_tokens': 37}, 'prompt_time': 0.021059483, 'prompt_tokens_details': None, 'queue_time': 0.409250204, 'total_time': 0.199304731}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_e5b4e54fbb', 'service_tier':

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result["messages"][-1].content))